# Step 3 · Does a better cycling network go with more cycling? (11 cities, 2017–2022)

**Question:** can the two infrastructure metrics of step 1 explain the two-wheeler commuting share of step 2,
and what share does the model predict for 2026?

**Model.** For city *c*, the infrastructure index mixes network length *N* and weighted connectivity *C*:

$$I = w_N\,N + (1-w_N)\,C,\qquad
\text{predicted share } p = 0.5\cdot\text{logistic}(\alpha_c+\beta_c I)=\frac{0.5}{1+e^{-(\alpha_c+\beta_c I)}}$$

The **50 % ceiling** is a modelling assumption (the share can approach it but never exceed it). The model is fitted
as a *fractional logit* (binomial GLM on share / 0.5). Three versions are compared:

* **A** — common intercept and slope for all cities;
* **B** — one intercept per city, common slope (**the model used in the rest of the study**);
* **C** — one intercept and one slope per city.

The weight $w_N$ (0, 0.05, …, 1) and the version are chosen by **leave-one-year-out cross-validation**: each
year 2017–2022 is removed in turn for all cities, the model is fitted on the other years and its error on the
removed year is measured. **2026 is never used for fitting**; it is a scenario prediction.

| | |
|---|---|
| **Input** | step 1 metrics (`results/lts_networks/<city>/…_metrics.txt`) and step 2 shares (`data/insee_modal_shares/`) |
| **Output** | `results/calibration/` — panel, CV scores, coefficients, fitted values, 2026 predictions |
| **Run time** | ~10 seconds |

> **Caution.** This is an association across cities and years, not a proof of causality. The dependent variable is
> INSEE *Deux-roues total* (bicycles + motorbikes), see step 2.

Code: [`cyclelts/mode_share.py`](../cyclelts/mode_share.py).

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from cyclelts import config, insee
from cyclelts.metrics import find_metrics_file, read_metrics_file, annual_metrics
from cyclelts.mode_share import (MAX_SHARE, STRUCTURES, with_weight, design, fit_model,
                                 predict_share, city_parameters)

MODE_COLUMN = "Deux-roues total"
WEIGHT_GRID = np.round(np.arange(0, 1.001, 0.05), 2)
OUTPUT = config.CALIBRATION_DIR
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Load the panel: one row per city and year

The metrics file of each city is the latest `fresh_*` run of step 1 (the TXT version, as in the published
calibration). Mode shares are converted from % to fractions.

In [3]:
def read_share(city):
    """Two-wheeler share (fraction) per year for one city."""
    df = insee.read_modal_shares(config.INSEE_SHARES_DIR, city.slug).rename(columns={"Year": "year"})
    y = pd.to_numeric(df[MODE_COLUMN].astype(str).str.replace("%", "", regex=False)
                      .str.replace(",", ".", regex=False), errors="coerce")
    if y.notna().any() and y.max() > 1:
        y = y / 100
    return pd.DataFrame({"year": df.year.astype(int), "share": y}).dropna()

rows, sources = [], []
for city in config.CITIES:
    mf = find_metrics_file(config.LTS_DIR / city.slug)
    merged = read_metrics_file(mf).merge(read_share(city), on="year", how="left")
    merged["city"] = city.name
    rows.append(merged)
    sources.append({"city": city.name, "metrics_file": str(mf.relative_to(ROOT))})
display(pd.DataFrame(sources))

panel = pd.concat(rows, ignore_index=True).sort_values(["city", "year"])
history = panel[panel.year.between(2017, 2022)].dropna(subset=["share"]).copy()
future = panel[panel.year == 2026].copy()
assert history.share.between(0, MAX_SHARE).all(), "Historical share outside [0, 50%]"
CITY_ORDER = sorted(history.city.unique())
print("Historical observations:", len(history), "| cities:", history.city.nunique(), "| 2026 rows:", len(future))
display(history.groupby("city").agg(first_year=("year", "min"), last_year=("year", "max"), observations=("year", "size")))

,city,metrics_file
0,Angers,results\lts_networks\angers\fresh_2026-10-21\a...
1,Annecy,results\lts_networks\annecy\fresh_2026-10-21\a...
2,Bordeaux,results\lts_networks\bordeaux\fresh_2026-10-21...
3,Chambéry,results\lts_networks\chambéry\fresh_2026-10-21...
4,Grenoble,results\lts_networks\grenoble\fresh_2026-10-21...
5,Lyon,results\lts_networks\lyon\fresh_2026-10-21\lyo...
6,Nantes,results\lts_networks\nantes\fresh_2026-10-21\n...
7,Paris,results\lts_networks\paris\fresh_2026-10-21\pa...
8,Rennes,results\lts_networks\rennes\fresh_2026-10-21\r...
9,Strasbourg,results\lts_networks\strasbourg\fresh_2026-10-...


Historical observations: 66 | cities: 11 | 2026 rows: 11


,first_year,last_year,observations
city,,,
Angers,2017,2022,6
Annecy,2017,2022,6
Bordeaux,2017,2022,6
Chambéry,2017,2022,6
Grenoble,2017,2022,6
Lyon,2017,2022,6
Nantes,2017,2022,6
Paris,2017,2022,6
Rennes,2017,2022,6


## 2. What the data look like

Each point is one city in one year. Both metrics are normalised by each city's own reference length, so they are
comparable between cities of different sizes.

In [4]:
fig = go.Figure()
for city, g in history.groupby("city"):
    fig.add_scatter(x=g.weighted_lcc_normalized, y=100 * g.share, mode="markers+lines", name=city,
                    text=g.year, hovertemplate="%{text}: C=%{x:.3f}, share=%{y:.1f}%")
fig.update_layout(template="plotly_white", height=480, title="Two-wheeler share vs weighted connectivity C, 2017–2022",
                  xaxis_title="Weighted largest connected component C (normalised)", yaxis_title="Two-wheeler share (%)")
fig.show()

## 3. Leave-one-year-out cross-validation over weights and model versions

The selected weight minimises the held-out RMSE (in percentage points). Because the weight is chosen with the same
data, the minimum CV error is slightly optimistic.

In [5]:
cv_rows = []
for structure_label, structure in STRUCTURES.items():
    for w in WEIGHT_GRID:
        data = with_weight(history, w)
        fold_preds, failures = [], []
        for test_year in sorted(data.year.unique()):
            tr, te = data[data.year != test_year], data[data.year == test_year]
            try:
                fit, _ = fit_model(tr, structure, CITY_ORDER)
                temp = te[["city", "year", "share"]].copy()
                temp["prediction"] = predict_share(fit, te, structure, CITY_ORDER)
                fold_preds.append(temp)
            except Exception as e:
                failures.append(f"{test_year}: {e}")
        if failures or not fold_preds:
            cv_rows.append({"structure": structure_label, "weight_length": w, "weight_lcc": 1-w,
                            "cv_rmse_pp": np.nan, "cv_mae_pp": np.nan, "status": "; ".join(failures)})
            continue
        err = 100 * (pd.concat(fold_preds).prediction - pd.concat(fold_preds).share)
        cv_rows.append({"structure": structure_label, "weight_length": w, "weight_lcc": 1-w,
                        "cv_rmse_pp": float(np.sqrt(np.mean(err**2))), "cv_mae_pp": float(np.mean(np.abs(err))),
                        "status": "ok"})
cv_results = pd.DataFrame(cv_rows).sort_values("cv_rmse_pp")
best_per_structure = cv_results.dropna(subset=["cv_rmse_pp"]).drop_duplicates("structure")
display(best_per_structure[["structure", "weight_length", "weight_lcc", "cv_rmse_pp", "cv_mae_pp"]].round(4))
selection = best_per_structure.iloc[0]
print("Lowest CV error:", selection[["structure", "weight_length", "cv_rmse_pp"]].to_dict())

fig = go.Figure()
for label in STRUCTURES:
    sub = cv_results[cv_results.structure == label].sort_values("weight_length")
    fig.add_scatter(x=sub.weight_length, y=sub.cv_rmse_pp, mode="lines+markers", name=label)
fig.update_layout(template="plotly_white", height=450, title="Cross-validation: choice of the length vs LCC weight",
                  xaxis_title="Weight on normalized length w_N (LCC weight = 1 − w_N)",
                  yaxis_title="Held-out RMSE (percentage points)")
fig.show()

,structure,weight_length,weight_lcc,cv_rmse_pp,cv_mae_pp
55,City intercepts + city slopes,0.65,0.35,0.8798,0.7459
29,City intercepts + common slope,0.40,0.60,1.0518,0.8266
0,Common intercept + slope,0.00,1.00,2.9098,2.1836


Lowest CV error: {'structure': 'City intercepts + city slopes', 'weight_length': 0.65, 'cv_rmse_pp': 0.8797586981535688}


## 4. Final coefficients (fitted on all 2017–2022 data)

Each version is fitted at its own best weight. `alpha` and `beta` are the city-specific intercept and slope on
the logit scale, directly usable in the formula at the top. The study keeps **"City intercepts + common slope"**:
almost the same CV error as the most flexible version, with far fewer parameters (one slope for all cities).

In [6]:
models, parameter_rows, historical_rows, forecast_rows = {}, [], [], []
for label, structure in STRUCTURES.items():
    chosen = best_per_structure[best_per_structure.structure == label].iloc[0]
    w = float(chosen.weight_length)
    train = with_weight(history, w)
    fit, names = fit_model(train, structure, CITY_ORDER)
    models[label] = {"fit": fit, "structure": structure, "weight_length": w}
    h = train[["city", "year", "share", "index"]].copy()
    h["predicted_share"] = predict_share(fit, train, structure, CITY_ORDER)
    h["model"] = label
    historical_rows.append(h)
    f = with_weight(future, w)[["city", "year", "index"]].copy()
    f["predicted_share"] = predict_share(fit, with_weight(future, w), structure, CITY_ORDER)
    f["model"] = label
    forecast_rows.append(f)
    for city, (alpha, beta) in city_parameters(fit, structure, CITY_ORDER).items():
        parameter_rows.append({"model": label, "city": city, "alpha": alpha, "beta": beta, "length_weight": w,
                               "lcc_weight": 1-w, "cv_rmse_pp": float(chosen.cv_rmse_pp)})
parameters = pd.DataFrame(parameter_rows)
historical_predictions = pd.concat(historical_rows, ignore_index=True)
forecast_2026 = pd.concat(forecast_rows, ignore_index=True)
display(parameters[parameters.model == "City intercepts + common slope"].round(5))

,model,city,alpha,beta,length_weight,lcc_weight,cv_rmse_pp
11,City intercepts + common slope,Angers,-3.77896,5.01498,0.4,0.6,1.05177
12,City intercepts + common slope,Annecy,-3.75427,5.01498,0.4,0.6,1.05177
13,City intercepts + common slope,Bordeaux,-3.04932,5.01498,0.4,0.6,1.05177
14,City intercepts + common slope,Chambéry,-3.93754,5.01498,0.4,0.6,1.05177
15,City intercepts + common slope,Grenoble,-3.40675,5.01498,0.4,0.6,1.05177
16,City intercepts + common slope,Lyon,-3.81956,5.01498,0.4,0.6,1.05177
17,City intercepts + common slope,Nantes,-3.66075,5.01498,0.4,0.6,1.05177
18,City intercepts + common slope,Paris,-4.11727,5.01498,0.4,0.6,1.05177
19,City intercepts + common slope,Rennes,-3.68089,5.01498,0.4,0.6,1.05177
20,City intercepts + common slope,Strasbourg,-3.44885,5.01498,0.4,0.6,1.05177


## 5. Observed vs fitted shares, and the 2026 prediction

Lines are fitted values on the calibration years (not cross-validated predictions); diamonds are the 2026
predictions from the 2026 network, an extrapolation in time.

In [7]:
def plot_city(city):
    fig = go.Figure()
    obs = history[history.city == city].sort_values("year")
    fig.add_scatter(x=obs.year, y=100*obs.share, name="INSEE observed", mode="lines+markers", line=dict(width=3, color="black"))
    for label in STRUCTURES:
        h = historical_predictions[(historical_predictions.city == city) & (historical_predictions.model == label)].sort_values("year")
        fig.add_scatter(x=h.year, y=100*h.predicted_share, name=label, mode="lines+markers")
        f = forecast_2026[(forecast_2026.city == city) & (forecast_2026.model == label)]
        fig.add_scatter(x=f.year, y=100*f.predicted_share, name=f"{label} — 2026", mode="markers",
                        marker=dict(size=12, symbol="diamond"))
    fig.update_layout(template="plotly_white", height=420, title=f"{city}: fitted values and 2026 prediction",
                      xaxis_title="Year", yaxis_title="Two-wheeler share (%)", yaxis=dict(range=[0, 50]))
    fig.show()

for city in CITY_ORDER:
    plot_city(city)

pivot = forecast_2026.pivot(index="city", columns="model", values="predicted_share") * 100
pivot.columns = [f"{c} (%)" for c in pivot.columns]
print("2026 predicted two-wheeler share (%):")
display(pivot.round(3))

2026 predicted two-wheeler share (%):


,City intercepts + city slopes (%),City intercepts + common slope (%),Common intercept + slope (%)
city,,,
Angers,13.918,14.682,14.097
Annecy,11.662,13.412,11.604
Bordeaux,24.826,23.838,15.087
Chambéry,13.180,12.594,13.196
Grenoble,24.110,22.820,20.553
Lyon,18.936,15.960,17.093
Nantes,17.058,16.582,15.154
Paris,18.116,17.670,26.619
Rennes,24.456,16.973,16.189


## 6. Save everything (audit trail)

In [8]:
panel.to_csv(OUTPUT / "city_infrastructure_mode_panel.csv", index=False)
cv_results.to_csv(OUTPUT / "weight_model_LOYO_CV.csv", index=False)
parameters.to_csv(OUTPUT / "city_model_parameters.csv", index=False)
historical_predictions.to_csv(OUTPUT / "historical_fitted_predictions.csv", index=False)
forecast_2026.to_csv(OUTPUT / "predictions_2026_all_models.csv", index=False)
report = [
    "LTS model calibration / 11 cities (2017–2022). 2026 excluded from fitting and CV.",
    "Dependent variable: INSEE Deux-roues total (check bicycle-only interpretation).",
    "Maximum predicted share: 50% (assumed).",
    "Index = w_length * normalized_network_length + (1-w_length) * weighted_lcc_normalized.",
    "Fractional logit: predicted_share = 0.5 * sigmoid(alpha_city + beta_city * index).",
    "Selection: leave-one-calendar-year-out CV, RMSE in percentage points.", "",
    "Best model per structure:", best_per_structure.to_string(index=False), "",
    "Selected overall:", selection.to_string(), "", "2026 scenario predictions:",
    forecast_2026.to_string(index=False),
    "", "Note: selected CV minimum can be optimistic; 2026 predictions are extrapolations, not observed.",
]
(OUTPUT / "calibration_summary.txt").write_text("\n".join(report), encoding="utf-8")
print("Saved in", OUTPUT.relative_to(ROOT))

Saved in results\calibration


## 7. Check: Grenoble 2026 = 22.820 %

The optimisation notebooks (steps 4–7) start from Grenoble's 2026 network and this model
(`cyclelts.mode_share.GRENOBLE_MODEL`). Here the metrics are recomputed **from the network files** and the
prediction is compared with the constant used downstream.

In [9]:
from cyclelts.mode_share import GRENOBLE_MODEL, GRENOBLE_2026_SHARE

graphs = {year: config.grenoble_graph(year) for year in config.ALL_YEARS}
recomputed = annual_metrics(graphs, kappa=1.0)
check = recomputed[["year", "normalized_network_length", "weighted_lcc_normalized"]].merge(
    panel.loc[panel.city == "Grenoble", ["year", "normalized_network_length", "weighted_lcc_normalized"]],
    on="year", suffixes=("", "_file"))
assert np.allclose(check.normalized_network_length, check.normalized_network_length_file, atol=1e-6)
assert np.allclose(check.weighted_lcc_normalized, check.weighted_lcc_normalized_file, atol=1e-6)

m = models["City intercepts + common slope"]
g = with_weight(recomputed.assign(city="Grenoble"), m["weight_length"])
g["predicted_pct"] = 100 * predict_share(m["fit"], g, m["structure"], CITY_ORDER)
g = g.merge(history.loc[history.city == "Grenoble", ["year", "share"]], on="year", how="left")
g["observed_pct"] = 100 * g.share
display(g[["year", "normalized_network_length", "weighted_lcc_normalized", "index", "predicted_pct", "observed_pct"]].round(4))
alpha, beta = parameters.query("model == 'City intercepts + common slope' and city == 'Grenoble'")[["alpha", "beta"]].iloc[0]
p2026 = float(g.loc[g.year == 2026, "predicted_pct"].iloc[0])
print(f"w_N = {m['weight_length']:.2f}, alpha_Grenoble = {alpha:.5f}, beta = {beta:.5f}")
print(f"Grenoble 2026 predicted share: {p2026:.3f} %  (constant used downstream: {GRENOBLE_2026_SHARE:.3f} %)")
assert abs(p2026 - GRENOBLE_2026_SHARE) < 0.005
assert abs(alpha - GRENOBLE_MODEL.alpha) < 5e-6 and abs(beta - GRENOBLE_MODEL.beta) < 5e-6
g.drop(columns="share").to_csv(OUTPUT / "grenoble_2017_2026_from_lts_output_graphs.csv", index=False)

,year,normalized_network_length,weighted_lcc_normalized,index,predicted_pct,observed_pct
0,2017,0.9301,0.3235,0.5662,18.0914,17.1949
1,2018,0.9389,0.3262,0.5713,18.3892,18.2061
2,2019,0.9652,0.3352,0.5872,19.3271,18.1690
3,2020,0.9684,0.3587,0.6026,20.2482,19.4701
4,2021,0.9886,0.3722,0.6187,21.2309,22.1922
5,2022,0.9933,0.3745,0.6220,21.4322,23.4867
6,2026,1.0000,0.4074,0.6445,22.8201,NaN


w_N = 0.40, alpha_Grenoble = -3.40675, beta = 5.01498
Grenoble 2026 predicted share: 22.820 %  (constant used downstream: 22.820 %)
